# Binance futures exogenous-feature screens

This notebook validates the durable evidence artifact for the fixed taker-flow and lagged-positioning screens. It does not authorize strategy integration: taker flow is negative, while positioning reverses sign in the second chronological half and has bootstrap intervals crossing zero.

In [1]:
import json
from pathlib import Path

ROOT = Path('/Users/ttoomm/Documents/PolyMomentum')
EVIDENCE = ROOT / 'deploy/promotions/evidence/strategy_registry/20260715_binance_futures_exogenous_screens.json'
payload = json.loads(EVIDENCE.read_text())
screens = {screen['name']: screen for screen in payload['screens']}
sorted(screens)

['lagged_position_crowding', 'prior_closed_minute_taker_flow']

In [2]:
flow = screens['prior_closed_minute_taker_flow']
positioning = screens['lagged_position_crowding']
assert payload['status'] == 'KEEP_REPLAY_RESEARCH'
assert payload['live_ready'] is False
assert flow['coverage']['complete_fraction'] == 1.0
assert positioning['coverage']['complete_fraction'] >= 0.99
assert flow['coverage']['timestamp_violations'] == 0
assert positioning['coverage']['timestamp_violations'] == 0
{'flow_rows': flow['coverage']['complete_rows'], 'positioning_rows': positioning['coverage']['complete_rows'], 'checksums': payload['source']['checksum_contract']}

{'flow_rows': 10609,
 'positioning_rows': 10555,
 'checksums': 'Every archive was verified against its adjacent Binance .CHECKSUM file.'}

In [3]:
comparisons = []
for label, screen in [('Taker flow', flow), ('Positioning', positioning)]:
    comparisons.append({
        'screen': label,
        'overall_brier': screen['overall']['market_minus_meta_brier'],
        'overall_log_loss': screen['overall']['market_minus_meta_log_loss'],
        'first_brier': screen['chronological_halves']['first_market_minus_meta_brier'],
        'second_brier': screen['chronological_halves']['second_market_minus_meta_brier'],
    })
comparisons

[{'screen': 'Taker flow',
  'overall_brier': -0.000785,
  'overall_log_loss': -0.002098,
  'first_brier': -0.001339,
  'second_brier': -0.000244},
 {'screen': 'Positioning',
  'overall_brier': 0.0017,
  'overall_log_loss': 0.005975,
  'first_brier': 0.004247,
  'second_brier': -0.000786}]

In [4]:
assert flow['overall']['market_minus_meta_brier'] < 0
assert flow['overall']['market_minus_meta_log_loss'] < 0
assert positioning['overall']['market_minus_meta_brier'] > 0
assert positioning['overall']['market_minus_meta_log_loss'] > 0
assert positioning['chronological_halves']['second_market_minus_meta_brier'] < 0
assert positioning['chronological_halves']['second_market_minus_meta_log_loss'] < 0
assert positioning['condition_bootstrap_95pct']['market_minus_meta_brier'][0] < 0
assert positioning['condition_bootstrap_95pct']['market_minus_meta_log_loss'][0] < 0
assert positioning['exact_trade_diagnostic']['mean_probability_correction_losses'] > 0
model = positioning['frozen_forward_model']
assert model['ridge'] == 10.0
assert max(abs(value) for value in model['standardized_coefficients'].values()) < 0.01
'All fail-closed assertions passed; no strategy integration or exact replay is authorized.'

'All fail-closed assertions passed; no strategy integration or exact replay is authorized.'

## Interpretation and boundary

The aggregate positioning improvement is hypothesis evidence only. It is concentrated in the first half, is not bootstrap-secure, and the final prior-fold ridge selection shrinks both feature coefficients close to zero. The model may be scored prospectively on a new fully resolved block, but it must not alter entries, sizing, or live controls until two disjoint fresh blocks pass the frozen gates.